#### Phase 3A–3B: CIF Matching and Metal-Node Descriptors

This notebook matches Phase 2 MOF records to CIF structure files and extracts initial metal-node descriptors for Phase 3.

** The logic for phase 3 is **
``` text 
Full raw folder:
32,768 JSON
32,767 CIF

Phase 2 cleaned modeling dataset:
~25,929 MOFs

Phase 3 base:
same ~25,929 MOFs
+ matching CIF path
+ metal / structure descriptors
```

---

In [ ]:
import sys
from pathlib import Path

# Add the project root so the shared path configuration is importable.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "paths.py").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import DATA_DIR, MODELS_DIR, RAW_DATA_DIR, RESULTS_DIR, RESULTS_ROOT


In [1]:
import pandas as pd 
import numpy as np
from pathlib import Path

In [2]:
# Define the folder that contains the raw hMOF JSON and CIF files.

raw_structure_dir = Path(RAW_DATA_DIR / "hmof_json_local")
# Check whether the raw structure folder exists.

print(raw_structure_dir.exists())
# Count how many CIF and JSON files are in the raw structure folder.

cif_files = list(raw_structure_dir.glob("*.cif"))
json_files = list(raw_structure_dir.glob("*.json"))

print("Number of CIF files:", len(cif_files))
print("Number of JSON files:", len(json_files))

True
Number of CIF files: 32767
Number of JSON files: 32768


In [3]:
# Define the path to the cleaned Phase 2 linker-extracted CSV dataset.

phase2_path = DATA_DIR / 'hmof_linker_extended_descriptors.csv'


# Load the cleaned Phase 2 dataset as the base dataset for Phase 3.
df = pd.read_csv(phase2_path)

# Check the key identifier columns in the cleaned Phase 2 dataset.
# These columns will help us match each Phase 2 MOF to its CIF file.

print(df[["filename", "mofid"]].head(10))
print()
print('='*30)

# Check whether the filename values already include file extensions.
df["filename"].head(20).tolist()



     filename                                              mofid
0      hMOF-0  [O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...
1      hMOF-1  [O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...
2     hMOF-10  [O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Z...
3    hMOF-100  COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1...
4   hMOF-1000  CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=...
5  hMOF-10000  [O-]C(=O)c1cc(Br)c2c(c1)ccc(c2)C(=O)[O-].[O-]C...
6  hMOF-10001  [O-]C(=O)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)ccc(c2C...
7  hMOF-10003  [O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]...
8  hMOF-10004  [O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]...
9   hMOF-1001  CCc1cc(C(=O)[O-])c(c(c1[C][O])CC)CC.[O].[O-]C(...



/tmp/ipykernel_10981/86672580.py:7: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(phase2_path)


['hMOF-0',
 'hMOF-1',
 'hMOF-10',
 'hMOF-100',
 'hMOF-1000',
 'hMOF-10000',
 'hMOF-10001',
 'hMOF-10003',
 'hMOF-10004',
 'hMOF-1001',
 'hMOF-10010',
 'hMOF-10011',
 'hMOF-10012',
 'hMOF-10013',
 'hMOF-10014',
 'hMOF-10015',
 'hMOF-10016',
 'hMOF-10017',
 'hMOF-10018',
 'hMOF-10019']

---

**Conclusion**: The filename column contains names such as hMOF-0 without the .cif extension, so we need to add .cif before matching to CIF files.

---

In [4]:
# Create the expected CIF filename for each MOF by adding the .cif extension.
# This is needed because the Phase 2 filename column contains names like hMOF-0, not hMOF-0.cif.

df["cif_filename"] = df["filename"].astype(str)+ ".cif"
# adding astype(str) helps us to make sure every value in df["filename"] is treated as text before adding
# A new cif_filename column was created by adding .cif to each MOF filename.

# Create the full CIF file path for each MOF using the raw structure folder.
# This prepares the exact location where Python should look for each CIF file.

# Define a function to create the full CIF file path from a CIF filename.
# This makes the code easier to read and easier to reuse later.

def make_cif_path(cif_filename):
    """
    Create the full CIF file path using the raw structure folder.

    Parameters
    ----------
    cif_filename : str
        Name of the CIF file, for example hMOF-0.cif.

    Returns
    -------
    pathlib.Path
        Full path to the CIF file.
    """
    return raw_structure_dir / cif_filename

# Apply the CIF path function to every CIF filename in the dataset.
# The cif_path column now stores the full expected file path for each CIF structure.
df["cif_path"] = df["cif_filename"].apply(make_cif_path)



In [5]:
# Define a function to check whether a CIF file exists at the expected path.
# This helps us identify which Phase 2 MOFs have matching CIF structure files.

def check_cif_exists(cif_path):
    """
    Check whether a CIF file exists at the given path.

    Parameters
    ----------
    cif_path : pathlib.Path
        Full expected path to the CIF file.

    Returns
    -------
    bool
        True if the CIF file exists, False otherwise.
    """
    return cif_path.exists()
# Apply the CIF existence check to every CIF path in the dataset.

df["cif_found"] = df["cif_path"].apply(check_cif_exists)

# Summarize how many Phase 2 MOFs have matching CIF files.
# This shows the CIF matching success rate for the Phase 3 base dataset.

print(df["cif_found"].value_counts())
print("Matched percent:", df["cif_found"].mean() * 100)

cif_found
True    25928
Name: count, dtype: int64
Matched percent: 100.0


---

Conclusion: All 25,928 cleaned Phase 2 MOFs were successfully matched to CIF files, giving a 100% matched subset for Phase 3 descriptor extraction.

---

In [6]:
# Preview the CIF matching columns to confirm that filenames and paths were created correctly.

print(df[["filename", "cif_filename", "cif_path", "cif_found"]].head(10))
print()
print('Conclusion: The CIF filename and full CIF path were created correctly for the matched Phase 3 dataset.')

     filename    cif_filename  \
0      hMOF-0      hMOF-0.cif   
1      hMOF-1      hMOF-1.cif   
2     hMOF-10     hMOF-10.cif   
3    hMOF-100    hMOF-100.cif   
4   hMOF-1000   hMOF-1000.cif   
5  hMOF-10000  hMOF-10000.cif   
6  hMOF-10001  hMOF-10001.cif   
7  hMOF-10003  hMOF-10003.cif   
8  hMOF-10004  hMOF-10004.cif   
9   hMOF-1001   hMOF-1001.cif   

                                            cif_path  cif_found  
0  /home/susan/mof-co2-adsorption/data/raw/hmof_j...       True  
1  /home/susan/mof-co2-adsorption/data/raw/hmof_j...       True  
2  /home/susan/mof-co2-adsorption/data/raw/hmof_j...       True  
3  /home/susan/mof-co2-adsorption/data/raw/hmof_j...       True  
4  /home/susan/mof-co2-adsorption/data/raw/hmof_j...       True  
5  /home/susan/mof-co2-adsorption/data/raw/hmof_j...       True  
6  /home/susan/mof-co2-adsorption/data/raw/hmof_j...       True  
7  /home/susan/mof-co2-adsorption/data/raw/hmof_j...       True  
8  /home/susan/mof-co2-adsorption/data/raw

In [7]:
# Save the Phase 3 CIF-matched dataset.
# This file will be used as the starting point for metal-node descriptor extraction.

matched_output_path = DATA_DIR / "hmof_phase3_cif_matched.csv"

df.to_csv(matched_output_path, index=False)

print("Saved:", matched_output_path)
print("Dataset shape:", df.shape)
print()
print()
print('Conclusion: The CIF-matched Phase 3 dataset was saved and is ready for metal-node descriptor extraction.')

Saved: /home/susan/mof-co2-adsorption/data/processed/hmof_phase3_cif_matched.csv
Dataset shape: (25928, 59)


Conclusion: The CIF-matched Phase 3 dataset was saved and is ready for metal-node descriptor extraction.



#### 3B — Extract Metal-Node Descriptors

This section extracts reproducible metal-node descriptors from each matched CIF structure, including metal identity, metal counts, and selected elemental properties.
**Goal:**  
Extract metal-node descriptors from matched CIF structures, including metal identity, metal counts, selected elemental properties, and exploratory oxidation-state/open-metal-site indicators.

**Why this matters:**  
Metal nodes can strongly influence CO₂ adsorption, especially at low pressure, because metal identity, charge environment, and coordination sites can affect CO₂ binding strength.

## Phase 3B Metal-Node Descriptor Plan

| Descriptor | Group | Meaning | Reliability |
|---|---|---|---|
| `metal_elements` | Metal identity/composition | List of metal elements present in each MOF, such as `Zn`, `Cu`, `Co`, or `Ni` | High |
| `num_metal_types` | Metal identity/composition | Number of distinct metal elements present in the structure | High |
| `total_metal_atoms` | Metal identity/composition | Total number of metal atoms in the CIF structure | High |
| `metal_fraction` | Metal identity/composition | Fraction of atoms in the structure that are metals | High |
| `avg_metal_atomic_number` | Elemental properties | Average atomic number of the metal atoms in the structure | High |
| `avg_metal_atomic_mass` | Elemental properties | Average atomic mass of the metal atoms in the structure | High |
| `estimated_oxidation_state` | Oxidation state | Estimated oxidation state of metal centers, if assignment is reliable | Investigate |
| `possible_open_metal_site` | Open metal sites | Indicator of possible under-coordinated metal centers, if assignment is reliable | Investigate |
| `total_atoms` | Metal identity/composition | Total number of atoms in the CIF structure; used to calculate metal fraction | High |


In [8]:
# Import tools for reading CIF structures and working with chemical elements.
# pymatgen will be used to parse each CIF file and identify elements in the structure.

from pymatgen.core import  Composition, Element, Lattice, Molecule, Structure

# Select the first matched CIF file to test whether pymatgen can read the structure.

test_cif_path = df.loc[0, "cif_path"]

print(test_cif_path)

/home/susan/mof-co2-adsorption/data/raw/hmof_json_local/hMOF-0.cif


In [9]:
# Read the sample CIF file with pymatgen.
# This checks whether pymatgen can successfully parse the structure before we process all CIF files.

test_structure = Structure.from_file(test_cif_path)
print(test_structure)
print()
print('pymatgen successfully parsed the sample CIF file, so this method can be used for metal descriptor extraction.')

Full Formula (Zn4 H12 C24 O13)
Reduced Formula: Zn4H12C24O13
abc   :  12.759401  12.759399  12.759393
angles:  89.967969  89.980163  90.016641
pbc   :       True       True       True
Sites (53)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  Zn    0.606252  0.581839  0.405983
  1  Zn    0.606145  0.40612   0.581664
  2  Zn    0.430554  0.58178   0.581648
  3  Zn    0.430541  0.406197  0.406027
  4  H     0.118467  0.371927  0.372148
  5  H     0.919336  0.616016  0.615482
  6  H     0.640379  0.372285  0.893778
  7  H     0.396672  0.616109  0.89376
  8  H     0.396543  0.894978  0.371966
  9  H     0.396629  0.616055  0.092807
 10  H     0.640467  0.895     0.615559
 11  H     0.118403  0.616041  0.61552
 12  H     0.39642   0.094047  0.372014
 13  H     0.91942   0.371922  0.372111
 14  H     0.640344  0.094048  0.615626
 15  H     0.640338  0.372212  0.092846
 16  C     0.245406  0.493997  0.493851
 17  C     0.128629  0.493977  0.493835
 18  C 

In [10]:
# Extract the elements present in the sample CIF structure.
# This helps us see whether the CIF contains metals and organic linker atoms.

test_elements =test_structure.composition.elements
print(test_elements)
print()
print('The sample CIF elements were extracted from the structure composition.')

[Element Zn, Element H, Element C, Element O]

The sample CIF elements were extracted from the structure composition.


In [11]:
# Identify the metal elements present in the sample CIF structure.
# This separates metal-node elements from non-metal linker atoms.

test_metal_elements = []
for element in test_elements:
    if element.is_metal:
        test_metal_elements.append(element)
print(test_metal_elements)

# or test_metal_elements = [element for element in test_elements if element.is_metal]
#print(test_metal_elements)

# Count the number of atoms for each element in the sample CIF structure.
# This lets us calculate the total number of metal atoms and total atoms.

test_composition = test_structure.composition
print(test_composition) 
print('The composition shows how many atoms of each element are present in the sample MOF.')
print('='*30)

# Count the number of atoms for each element in the sample CIF structure.
# This lets us calculate the total number of metal atoms and total atoms.
total_metal_atoms =0 
for metal in test_metal_elements:
    total_metal_atoms += test_composition[metal]
print(total_metal_atoms)
print('The sample MOF has 4 total metal atoms and 1 distinct metal type.')

[Element Zn]
Zn4 H12 C24 O13
The composition shows how many atoms of each element are present in the sample MOF.
4.0
The sample MOF has 4 total metal atoms and 1 distinct metal type.


| Descriptor | Meaning |
|---|---|
| `total_metal_atoms` | How many metal atoms are in the CIF |
| `num_metal_types` | How many different metals, e.g. only Zn = 1 |
| `total_atoms` | How large the CIF structure is |
| `metal_fraction` | How metal-rich the structure is |

In [12]:
# Count the total number of atoms in the sample CIF structure.
# This will be used later to calculate the fraction of atoms that are metals.
#nmetal_fraction = total_metal_atoms / total_atoms
total_atoms = test_composition.num_atoms
print(total_atoms)
print('The sample CIF structure contains 53 atoms in total.')
print('='*30)
# Check the metal atom count and total atom count before calculating metal fraction.
# This helps confirm that the fraction is being calculated from the correct values.


# Calculate the metal atom fraction in the sample CIF structure.
# This normalizes the metal atom count by the total number of atoms
test_metal_fraction =total_metal_atoms/total_atoms
print(test_metal_fraction)
print('The sample MOF has a metal fraction of about 0.075, meaning about 7.5% of its atoms are metals.')

53.0
The sample CIF structure contains 53 atoms in total.
0.07547169811320754
The sample MOF has a metal fraction of about 0.075, meaning about 7.5% of its atoms are metals.


In [13]:
# Define a function to extract basic metal-node descriptors from one CIF file.
# This function reads one CIF structure and calculates metal identity, counts, fraction, and atomic properties.

def extract_basic_metal_descriptors(cif_path):
    """
    Extract basic metal-node descriptors from one CIF structure.

    Parameters
    ----------
    cif_path : str or pathlib.Path
        Full path to the CIF file.

    Returns
    -------
    dict
        Dictionary containing metal elements, number of metal types,
        total metal atoms, total atoms, metal fraction,
        average metal atomic number, and average metal atomic mass.
    """

    structure = Structure.from_file(cif_path)
    composition = structure.composition
    elements = composition.elements

    metal_elements = []

    for element in elements:
        if element.is_metal:
            metal_elements.append(element)

    total_metal_atoms = 0

    for metal in metal_elements:
        total_metal_atoms += composition[metal]

    total_atoms = composition.num_atoms

    if total_atoms > 0:
        metal_fraction = total_metal_atoms / total_atoms
    else:
        metal_fraction = None

    # Calculate weighted average atomic number and atomic mass for metal atoms.
    # Weighting by atom count is important when a MOF contains more than one metal type.
    if total_metal_atoms > 0:
        weighted_atomic_number_sum = 0
        weighted_atomic_mass_sum = 0

        for metal in metal_elements:
            metal_count = composition[metal]
            weighted_atomic_number_sum += metal.Z * metal_count
            weighted_atomic_mass_sum += float(metal.atomic_mass) * metal_count

        avg_metal_atomic_number = weighted_atomic_number_sum / total_metal_atoms
        avg_metal_atomic_mass = weighted_atomic_mass_sum / total_metal_atoms
    else:
        avg_metal_atomic_number = None
        avg_metal_atomic_mass = None

    return {
        "metal_elements": [str(metal) for metal in metal_elements],
        "num_metal_types": len(metal_elements),
        "total_metal_atoms": total_metal_atoms,
        "total_atoms": total_atoms,
        "metal_fraction": metal_fraction,
        "avg_metal_atomic_number": avg_metal_atomic_number,
        "avg_metal_atomic_mass": avg_metal_atomic_mass,
    }
print()
print('A reusable function was created to extract basic metal-node descriptors\nfrom one CIF file.')




A reusable function was created to extract basic metal-node descriptors
from one CIF file.


In [14]:
# Test the updated metal descriptor function on one sample CIF.
# This confirms that the function returns the expected values for hMOF-0.

test_metal_descriptors = extract_basic_metal_descriptors(test_cif_path)

print(test_metal_descriptors)

{'metal_elements': ['Zn'], 'num_metal_types': 1, 'total_metal_atoms': 4.0, 'total_atoms': 53.0, 'metal_fraction': 0.07547169811320754, 'avg_metal_atomic_number': 30.0, 'avg_metal_atomic_mass': 65.409}


In [15]:
# Apply the metal descriptor function to the first 5 CIF files only.
# This small test checks that the function works across multiple structures before processing the full dataset.
test_descriptor_results = []
for cif_path in df['cif_path'].head(5):
    descriptors = extract_basic_metal_descriptors(cif_path)
    test_descriptor_results.append(descriptors)

print(test_descriptor_results)
print('The function was tested on the first 5 CIF files to confirm\n it works across multiple MOF structures.')
# Convert the first 5 metal descriptor results into a DataFrame.
# This shows how the extracted descriptors will look as table columns.

test_descriptor_df = pd.DataFrame(test_descriptor_results)
print('The first 5 descriptor results were converted into table columns, confirming that the function output can be merged back into the main dataset.')
test_descriptor_df

[{'metal_elements': ['Zn'], 'num_metal_types': 1, 'total_metal_atoms': 4.0, 'total_atoms': 53.0, 'metal_fraction': 0.07547169811320754, 'avg_metal_atomic_number': 30.0, 'avg_metal_atomic_mass': 65.409}, {'metal_elements': ['Zn'], 'num_metal_types': 1, 'total_metal_atoms': 8.0, 'total_atoms': 106.0, 'metal_fraction': 0.07547169811320754, 'avg_metal_atomic_number': 30.0, 'avg_metal_atomic_mass': 65.409}, {'metal_elements': ['Zn'], 'num_metal_types': 1, 'total_metal_atoms': 8.0, 'total_atoms': 106.0, 'metal_fraction': 0.07547169811320754, 'avg_metal_atomic_number': 30.0, 'avg_metal_atomic_mass': 65.409}, {'metal_elements': ['Zn'], 'num_metal_types': 1, 'total_metal_atoms': 4.0, 'total_atoms': 69.0, 'metal_fraction': 0.057971014492753624, 'avg_metal_atomic_number': 30.0, 'avg_metal_atomic_mass': 65.409}, {'metal_elements': ['Zn'], 'num_metal_types': 1, 'total_metal_atoms': 8.0, 'total_atoms': 110.0, 'metal_fraction': 0.07272727272727272, 'avg_metal_atomic_number': 30.0, 'avg_metal_atomic_m

,metal_elements,num_metal_types,total_metal_atoms,total_atoms,metal_fraction,avg_metal_atomic_number,avg_metal_atomic_mass
0,[Zn],1,4.0,53.0,0.075472,30.0,65.409
1,[Zn],1,8.0,106.0,0.075472,30.0,65.409
2,[Zn],1,8.0,106.0,0.075472,30.0,65.409
3,[Zn],1,4.0,69.0,0.057971,30.0,65.409
4,[Zn],1,8.0,110.0,0.072727,30.0,65.409


In [16]:
# Apply the metal descriptor function to all matched CIF files.
# This extracts metal-node descriptors for every MOF in the Phase 3 dataset.

metal_descriptor_results = []

for cif_path in df["cif_path"]:
    descriptors = extract_basic_metal_descriptors(cif_path)
    metal_descriptor_results.append(descriptors)

print("Number of descriptor rows:", len(metal_descriptor_results))

/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return parser.parse_structures(primitive=primitive)[0]
/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return parser.parse_structures(primitive=primitive)[0]
/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return parser.parse_structures(primitive=primitive)[0]
/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CI

Number of descriptor rows: 25928


In [17]:
# Convert the list of metal descriptor dictionaries into a DataFrame.
# This creates one descriptor row for each MOF.

metal_df = pd.DataFrame(metal_descriptor_results)
# Define the metal-node descriptor columns that will be added to the existing df.
# Keeping the column names in one list makes the code easier to check and reuse.

new_cols = [
    "metal_elements",
    "num_metal_types",
    "total_metal_atoms",
    "total_atoms",
    "metal_fraction",
    "avg_metal_atomic_number",
    "avg_metal_atomic_mass"
]
# Add the metal-node descriptor columns directly to the existing Phase 3 dataframe.
# The row order matches because metal descriptors were extracted using df["cif_path"] in order.

df[new_cols] = metal_df[new_cols]
print(" Metal-node descriptor columns were added directly to the existing Phase 3 dataframe.")

 Metal-node descriptor columns were added directly to the existing Phase 3 dataframe.


In [18]:
# Confirm that the number of rows in df and metal_df match before saving.
# This helps verify that no MOFs were missed during descriptor extraction.

print("Rows in df:", len(df))
print("Rows in metal_df:", len(metal_df))
print("Match:", len(df) == len(metal_df))
print("shape",df.shape)

Rows in df: 25928
Rows in metal_df: 25928
Match: True
shape (25928, 66)


In [19]:
# Print all dataframe column names.
# This helps check which columns are currently available in df.

for column in df.columns:
    print(column)

filename
lcd
pld
void_fraction
surface_area_m2g
mofid
CO2_uptake_0.01bar_molkg
CO2_uptake_0.05bar_molkg
CO2_uptake_0.1bar_molkg
CO2_uptake_0.5bar_molkg
CO2_uptake_2.5bar_molkg
chemical_representation
linker_smiles
invalid_fragments
linker_1
linker_2
linker_3
linker_4
linker_5
linker_6
linker_7
linker_8
linker_9
linker_10
linker_11
linker_12
linker_13
total_C_count
total_N_count
total_O_count
total_F_count
total_S_count
total_Cl_count
total_Br_count
total_I_count
total_heteroatom_count
total_heavy_atom_count
heteroatom_fraction
total_amine_count
total_hydroxyl_count
total_carboxylate_or_carboxylic_acid_count
total_fluorinated_group_count
total_sulfonyl_or_sulfate_count
total_MolWt
mean_MolWt
max_MolWt
total_TPSA
mean_TPSA
max_TPSA
total_HBD
total_HBA
total_aromatic_atom_count
mean_aromatic_fraction
total_heavy_atom_count_rdkit
total_ring_count
total_rotatable_bond_count
cif_filename
cif_path
cif_found
metal_elements
num_metal_types
total_metal_atoms
total_atoms
metal_fraction
avg_metal_

In [20]:
# Save the updated Phase 3 dataframe with metal-node descriptor columns.
# This file will be used as the starting point for Phase 3C.

phase3_metal_output_path = DATA_DIR / "hmof_phase3_metal_descriptors.csv"

df.to_csv(phase3_metal_output_path, index=False)

print("Saved:", phase3_metal_output_path)
print("Dataset shape:", df.shape)
print("Conclusion: The Phase 3B dataset with metal-node descriptors was saved successfully.")

Saved: /home/susan/mof-co2-adsorption/data/processed/hmof_phase3_metal_descriptors.csv
Dataset shape: (25928, 66)
Conclusion: The Phase 3B dataset with metal-node descriptors was saved successfully.
